# Findings

- MRO logic : How `TemporalBatchMixin` executes `forward`.
- `einops` : handling 5D tensors. 

In [23]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 


# `TemporalBatchMixin`

TemporalBatchMixin.forward runs because ResNet5 has no forward, so the MRO finds the mixin's. ✓ <br>
Inside it, self._forward resolves via the MRO to ResNet5._forward (the subclass's version shadows the mixin's). ✓

The mixin's _forward is a fallback that is only ever reached if a subclass forgot to define its own _forward. In normal operation — ResNet5 does define _forward — the mixin's _forward is completely shadowed and never executes at all. It's dead code in the happy path. It only becomes reachable when someone writes a subclass of the mixin but omits _forward; then self._forward falls through the MRO to the stub, which raises NotImplementedError to say "you were required to implement this and didn't."

# Einops

In [21]:
from einops import rearrange

t = torch.randint(0,10,(2,1,10,64,64))

mutate_t = rearrange(t, "b c t h w -> (b t) c h w")
print(mutate_t.shape) 

unmutate_t = rearrange(mutate_t, "(b t) c h w -> b c t h w", b = 2)
print(unmutate_t.shape)

torch.Size([20, 1, 64, 64])
torch.Size([2, 1, 10, 64, 64])


# `Projector`

In [46]:
mlp_spec = "4-2-3-5"

In [48]:
f = list(map(int,mlp_spec.split("-"))) 
f 

[4, 2, 3, 5]

In [49]:
layers = []

for i in range(len(f) -2): 
    layers.append(nn.Linear(f[i], f[i+1]))
    layers.append(nn.BatchNorm1d(f[i+1]))
    layers.append(nn.ReLU(True))

layers

[Linear(in_features=4, out_features=2, bias=True),
 BatchNorm1d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
 ReLU(inplace=True),
 Linear(in_features=2, out_features=3, bias=True),
 BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
 ReLU(inplace=True)]

In [50]:
layers.append(nn.Linear(f[-2] , f[-1], bias=False))
layers

[Linear(in_features=4, out_features=2, bias=True),
 BatchNorm1d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
 ReLU(inplace=True),
 Linear(in_features=2, out_features=3, bias=True),
 BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
 ReLU(inplace=True),
 Linear(in_features=3, out_features=5, bias=False)]

In [51]:
net = nn.Sequential(*layers)
net

Sequential(
  (0): Linear(in_features=4, out_features=2, bias=True)
  (1): BatchNorm1d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): Linear(in_features=2, out_features=3, bias=True)
  (4): BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (5): ReLU(inplace=True)
  (6): Linear(in_features=3, out_features=5, bias=False)
)